<a href="https://colab.research.google.com/github/syedbeeban/IE/blob/main/03_Regularization_for_Noisy_Integral_Equations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

When dealing with real-world data, the right-hand side of your integral equation (often representing sensor readings or experimental measurements) will contain noise.

This introduces a major challenge: many integral equations—especially **Fredholm equations of the first kind**—are highly sensitive to noise. A tiny measurement error can cause the optimization solver to return a wildly oscillating, incorrect function $u(x)$. This is known as an **ill-posed inverse problem**.

To fix this, we introduce **Regularization** (most commonly Tikhonov Regularization, also known as L2 regularization or Ridge regression). We modify our objective function to tell the solver: *"Find a solution that minimizes the error, but also keep the solution smooth and well-behaved."*

Here is how you add regularization to the optimization methods to handle noisy data.

---

### 1. Gradient-Based Optimization with Tikhonov Regularization

In traditional optimization, we add a penalty term to the objective function. This penalty is controlled by a parameter $\alpha$ (alpha).

* If $\alpha$ is too small, the noise ruins the solution.
* If $\alpha$ is too large, the solution becomes overly flat and ignores the data.

Let's simulate noisy sensor data for our right-hand side (RHS) and use regularization to stabilize the polynomial coefficients.

In [ ]:
import numpy as np
from scipy.optimize import minimize

x_points = np.linspace(0, 1, 20)
t_points = np.linspace(0, 1, 50)
dt = t_points[1] - t_points[0]

# Simulate noisy data for the Right Hand Side
exact_rhs = (5.0 / 6.0) * x_points
np.random.seed(42)
noise = np.random.normal(0, 0.05, size=exact_rhs.shape) # 5% Gaussian noise
noisy_rhs = exact_rhs + noise

def objective_regularized(c, alpha=0.1):
    """
    Minimizes the residual with an added Tikhonov regularization penalty.
    """
    total_error = 0
    u_t = c[0] + c[1]*t_points + c[2]*t_points**2

    # 1. Calculate the standard Data Residual
    for i, x in enumerate(x_points):
        u_x = c[0] + c[1]*x + c[2]*x**2
        integral_val = 0.5 * np.sum(x * t_points * u_t * dt)
        lhs = u_x - integral_val

        # Compare against the NOISY data
        total_error += (lhs - noisy_rhs[i])**2

    # 2. Add the Regularization Penalty
    # We penalize large coefficients to enforce a "smoother", less extreme curve
    # Penalty = alpha * (c0^2 + c1^2 + c2^2)
    penalty = alpha * np.sum(np.square(c))

    return total_error + penalty

# Run optimization with a chosen alpha
initial_guess = [0.0, 0.0, 0.0]
result_reg = minimize(objective_regularized, initial_guess, args=(0.05,), method='BFGS')

print("Regularized BFGS Results on Noisy Data:")
print(f"Optimized Coefficients: {result_reg.x}")
print("Notice how the coefficients stay close to [0, 1, 0] despite the noise, thanks to the penalty.")

---

### 2. Neural Networks (PINNs) with Weight Decay

In Deep Learning, Tikhonov regularization is mathematically equivalent to **Weight Decay**. Instead of manually writing the penalty into the loss function, modern deep learning frameworks allow you to apply this directly inside the optimizer. It penalizes large weights in the neural network, preventing it from over-fitting to the random fluctuations of the noisy data.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class NoisyIntegralNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 16),
            nn.Tanh(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        return self.net(x)

model = NoisyIntegralNet()

# Add Weight Decay (L2 Regularization) directly in the Adam Optimizer
# weight_decay = 1e-4 acts as our 'alpha' regularization parameter
optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)

x_tensor = torch.linspace(0, 1, 20).view(-1, 1)
t_tensor = torch.linspace(0, 1, 50).view(-1, 1)
dt = t_tensor[1] - t_tensor[0]

# Create noisy RHS tensor
exact_rhs_tensor = (5.0 / 6.0) * x_tensor
torch.manual_seed(42)
noisy_rhs_tensor = exact_rhs_tensor + torch.randn(exact_rhs_tensor.shape) * 0.05

epochs = 1000
for epoch in range(epochs):
    optimizer.zero_grad()

    u_x = model(x_tensor)
    u_t = model(t_tensor)

    K_matrix = x_tensor @ t_tensor.T
    integral_val = 0.5 * torch.sum(K_matrix * u_t.T * dt, dim=1).view(-1, 1)

    lhs = u_x - integral_val

    # Loss calculated against the NOISY data
    loss = torch.mean((lhs - noisy_rhs_tensor)**2)

    loss.backward()
    optimizer.step()

    if epoch % 500 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.6f}")

print("\nRegularized Neural Network Results:")
test_x = torch.tensor([[0.5], [1.0]])
predicted_u = model(test_x).detach().numpy()
print(f"Prediction at x=0.5: {predicted_u[0][0]:.4f} (Expected: 0.5)")

### Summary of Regularization Techniques

| Concept | Traditional Math Equivalent | Machine Learning Equivalent | Purpose |
| --- | --- | --- | --- |
| **L2 Penalty on Parameters** | Tikhonov Regularization (Ridge) | Weight Decay | Shrinks coefficients/weights to prevent the model from capturing random noise. |
| **L1 Penalty on Parameters** | LASSO Regularization | L1 Regularization | Forces some coefficients to exactly zero, creating "sparse" solutions. Good if you suspect the true solution only uses a few basis functions. |
| **Derivative Penalty** | Sobolev Regularization | Physics-Informed Gradient Penalty | Penalizes the *derivatives* of $u(x)$ directly in the loss function to enforce a physically smooth curve. |

Would you like to see how to implement **Derivative Penalty (Sobolev Regularization)** in the PyTorch model to force the neural network to prioritize smooth, physical shapes?